#  Data Acquisition & Automated ETL Pipeline
**Project:** Validation of the Richard A Ferri Hypothesis from the book **All About Asset Allocation** (the theory that gold’s high volatility and risk provide no real long-term growth beyond basic inflation tracking)  
**Objective:** To quantitatively validate the above theory by analyzing Gold as a long-term inflation hedge for EUR and INR investors through 2026.

> **Temporal Range:** Jan 2000 – Apr 2026 
---

###  Pipeline Architecture
This notebook executes a three-stage **Extract, Transform, Load (ETL)** process:

| Stage | Process | Component |
| :--- | :--- | :--- |
| **01** | **Asset Synthesis** | Merging Gold with S&P 500, Nifty 50, and DAX benchmarks. |
| **02** | **Currency Normalization** | Integrating USD/INR and USD/EUR spot rates for local-market parity. |
| **03** | **Macroeconomic Anchoring** | Harmonizing multi-base CPI indices for Real-Return modeling. |

##### **Key Technical Note:** All data is normalized to a daily frequency using `ffill` (Forward-Fill) to resolve international market holiday gaps.

---

!pip install python-dotenv

In [ ]:
import pandas as pd
import os

In [ ]:
from dotenv import load_dotenv

# Load the variables from .env
load_dotenv()

# Access the hidden path
data_dir = os.getenv('DATA_DIR')

## Step 1: Asset Integration (Gold & Equity Indices)
This section merges the historical data archives with modern market data to create a continuous timeline for Gold, the S&P 500, Nifty 50, and the DAX.

###  Data Sources
* **Gold:** `Investing.com` (2000-2022) + `Yahoo Finance` (2023-Present).
* **Equity Benchmarks:** `Yahoo Finance`- S&P 500 (USA), DAX (Germany), and Nifty 50 (India). 

###  Integration Logic
* **Temporal Splicing:** Merges legacy Nifty 50 and Gold archives to ensure data continuity from Jan 2000.
* **Alignment:** All indices are joined onto the Gold timeline and forward-filled to resolve regional market holidays.

In [ ]:
# --- 1. CONFIGURATION & FILE LOADING ---
base_file = os.path.join(data_dir, 'commodity_2000-2022.csv')
global_file = os.path.join(data_dir, 'global_financial_markets_2000_Now.csv')
nifty_old_file = os.path.join(data_dir, 'NIFTY 50_Historical_PR_01011990to11102024.csv')

df_base = pd.read_csv(base_file, parse_dates=['Date'])
df_global = pd.read_csv(global_file, parse_dates=['date'])
df_nifty_old = pd.read_csv(nifty_old_file)
df_nifty_old['Date'] = pd.to_datetime(df_nifty_old['Date'], format='mixed', dayfirst=True, errors='coerce')

# --- 2. GOLD TIMELINE SPLICING ---
gold_base = df_base[df_base['Symbol'].str.upper() == 'GOLD'].copy()
cutoff_date = gold_base['Date'].max()

gold_recent = df_global[(df_global['asset_name'] == 'Gold') & (df_global['date'] > cutoff_date)].copy()
gold_recent = gold_recent.rename(columns={'date': 'Date', 'close': 'Close', 'volume': 'Volume'})

gold_master = pd.concat([gold_base[['Date', 'Close', 'Volume']], gold_recent[['Date', 'Close', 'Volume']]], ignore_index=True)
gold_master = gold_master.sort_values('Date').drop_duplicates('Date')
final_df = gold_master.rename(columns={'Close': 'Gold_Close', 'Volume': 'Gold_Volume'})

# --- 3. EQUITY INDICES MERGING (S&P500, DAX, NIFTY50) ---
# US & German Indices from Global File
sp500 = df_global[df_global['asset_name'].isin(['S&P500', 'S&P 500'])][['date', 'close']].rename(columns={'date': 'Date', 'close': 'SP500'})
dax = df_global[df_global['symbol'] == '^GDAXI'][['date', 'close']].rename(columns={'date': 'Date', 'close': 'DAX'})

# Indian Index Splicing (Old Archive + Global Patch)
nifty_p1 = df_nifty_old[(df_nifty_old['Date'] >= '2000-01-01') & (df_nifty_old['Date'] <= '2007-09-17')][['Date', 'Close']]
nifty_p2 = df_global[(df_global['asset_name'] == 'NIFTY50') & (df_global['date'] > '2007-09-17')][['date', 'close']].rename(columns={'date': 'Date', 'close': 'Close'})
nifty_full = pd.concat([nifty_p1, nifty_p2]).rename(columns={'Close': 'Nifty50'})

# Merge all into final_df
final_df = final_df.merge(sp500, on='Date', how='left')
final_df = final_df.merge(dax, on='Date', how='left')
final_df = final_df.merge(nifty_full, on='Date', how='left')

# --- 4. DATA INTEGRITY ---
final_df = final_df.sort_values('Date')
target_cols = ['Gold_Close', 'Gold_Volume', 'SP500', 'DAX', 'Nifty50']
final_df[target_cols] = final_df[target_cols].ffill().bfill()

# --- 5. EXPORT ---
output_path = os.path.join(data_dir, 'financial_indices_master.csv')
final_df.to_csv(output_path, index=False)

print(f" Indices Master Created. Columns: {final_df.columns.tolist()}")

## Step 2: Currency Normalization (USD/INR & USD/EUR)
Integrates the exchange rates required to convert global Gold prices (USD) into local terms for Indian and German investors.

### Sourcing Strategy
* **USD/INR:** Sourced from `FRED (DEXINUS)`. Provides the authoritative exchange rate for the Indian Rupee.
* **USD/EUR:** Derived from `ECB/Eurostat` archives. 

### Engineering
* **Standardization:** All rates are converted to the `USD/Local` convention (i.e., cost of 1 USD in local currency).
* **Math:** Inverted spot rates ($1 / \text{rate}$) where necessary to ensure consistent multipliers for future Gold-to-Local price conversions.

In [ ]:
# --- 1. CONFIGURATION ---
eur_usd_path = os.path.join(data_dir, 'EURUSD.csv')
usd_inr_path = os.path.join(data_dir, 'USDINR.csv')

# Normalize master date to ensure perfect merging
final_df['Date'] = pd.to_datetime(final_df['Date']).dt.normalize()

# --- 2. USD/INR PIPELINE  ---
df_usd_inr = pd.read_csv(usd_inr_path)
df_usd_inr['Date'] = pd.to_datetime(df_usd_inr['observation_date'], format='mixed', dayfirst=True).dt.normalize()
usd_inr_final = df_usd_inr[['Date', 'DEXINUS']].rename(columns={'DEXINUS': 'USD_INR'})
usd_inr_final = usd_inr_final.drop_duplicates('Date')

# --- 3. USD/EUR PIPELINE (Inversion) ---
df_eur = pd.read_csv(eur_usd_path)
df_eur['Date'] = pd.to_datetime(df_eur['date'], format='mixed', dayfirst=True).dt.normalize()

# Convert EURUSD to USD/EUR (1 / rate)
df_eur['USD_EUR'] = 1 / df_eur['rate']
usd_eur_final = df_eur[['Date', 'USD_EUR']].drop_duplicates('Date')

# --- 4. MERGE & INTEGRITY ---
# Merge both currencies onto the master timeline
final_df = final_df.merge(usd_inr_final, on='Date', how='left')
final_df = final_df.merge(usd_eur_final, on='Date', how='left')

# Fill gaps (holidays/weekends)
final_df[['USD_INR', 'USD_EUR']] = final_df[['USD_INR', 'USD_EUR']].ffill().bfill()

# --- 5. EXPORT ---
final_output_path = os.path.join(data_dir, 'ULTIMATE_financial_master.csv')
final_df.to_csv(final_output_path, index=False)

print("Currency Normalization Complete using FRED and local archives.")
print(f"Final Columns: {final_df.columns.tolist()}")
print(f"Total Rows: {len(final_df)}")

## Step 3: Macroeconomic Integration (CPI & Inflation)
Final stage incorporating Consumer Price Index (CPI) data to facilitate "Real-Return" (inflation-adjusted) calculations.

### Data Source & Provenance
* **Germany:** Synthesis of `FRED` (Base 2015) and `Destatis` (Base 2020). 
* **India:** Sourced from `FRED` and patched via `RBI DBIE` (CPI-Combined).
* **United States:** `Alpha Vantage` archives patched with `BLS (Bureau of Labor Statistics)` data.

### Engineering: Harmonization & Temporal Alignment
* **German Link Factor (1.05428):** Used to bridge different base years.  
  * *Calculation:* $(\text{FRED Mar-2025: } 127.7792) / (\text{Destatis Mar-2025: } 121.2) = \mathbf{1.05428}$
* **Temporal Resampling:** Monthly CPI figures are mapped to daily records using a `Year-Month` period join and forward-filled to allow for daily "Real Value" indexing.

In [ ]:
# --- 1. LOAD & NORMALIZE MASTER ---
df_master = pd.read_csv(os.path.join(data_dir, 'ULTIMATE_financial_master.csv'))
df_master['Date'] = pd.to_datetime(df_master['Date'], format='mixed', dayfirst=True)
df_master['Year_Month'] = df_master['Date'].dt.to_period('M')

# --- 2. CONSOLIDATE CPI DATA (With Strict Deduplication) ---
# India
df_in = pd.read_csv(os.path.join(data_dir, 'india_inflation_master_final.csv'))
df_in['Year_Month'] = pd.to_datetime(df_in['Date'], format='mixed').dt.to_period('M')
# Use .groupby to ensure exactly one value per month
df_in = df_in.groupby('Year_Month')['CPI'].first().reset_index().rename(columns={'CPI': 'CPI_India'})

# USA
df_us = pd.read_csv(os.path.join(data_dir, 'CPI_USA.csv'))
df_us['Year_Month'] = pd.to_datetime(df_us['timestamp'], format='mixed').dt.to_period('M')
# Taking the first/mean value per month to stop the row explosion
df_us = df_us.groupby('Year_Month')['value'].mean().reset_index().rename(columns={'value': 'CPI_USA'})

# Germany
df_de = pd.read_csv(os.path.join(data_dir, 'CPI_DE.csv'))
df_de['Year_Month'] = pd.to_datetime(df_de.iloc[:, 0], format='mixed', dayfirst=True).dt.to_period('M')
df_de = df_de.groupby('Year_Month')[df_de.columns[1]].first().reset_index().rename(columns={df_de.columns[1]: 'CPI_Germany'})

# --- 3. SINGLE-PASS MERGE ---
cpi_lookup = df_in.merge(df_us, on='Year_Month', how='outer').merge(df_de, on='Year_Month', how='outer')

# Join to daily master
final_df = df_master.merge(cpi_lookup, on='Year_Month', how='left')

# --- 4. CLEANUP ---
final_df = final_df.drop(columns=['Year_Month'])
# Standard forward fill for the daily slots
final_df[['CPI_India', 'CPI_USA', 'CPI_Germany']] = final_df[['CPI_India', 'CPI_USA', 'CPI_Germany']].ffill().bfill()

# --- 5. EXPORT ---
output_path = os.path.join(data_dir, 'ULTIMATE_financial_master_WITH_CPI.csv')
final_df.to_csv(output_path, index=False)

print(f" Row explosion fixed. Total rows should now match original master: {len(df_master)}")
print(final_df.head())